In [0]:
%pip install openai mlflow

## Hosting external model endpoint
1. create an external model endpoint. This is simply wrapping an external model API call in an `mlflow pyfunc`
2. Make a call to that and pass in additional requests in the body

In [0]:
import mlflow.deployments

client = mlflow.deployments.get_deploy_client("databricks")
client.create_endpoint(
    name="openai-embeddings-iherb-test",
    config={
        "served_entities": [{
            "name": "text-embedding-3-small",
            "external_model": {
                "name": "text-embedding-3-small",
                "provider": "openai",
                "task": "llm/v1/embeddings",
                "openai_config": {
                    "openai_api_key": "{{secrets/notebooks/jon_openai_api_key}}"
                }
            }
        }]
    }
)

In [0]:
import mlflow.deployments

# client = mlflow.deployments.get_deploy_client("databricks")

completions_response = client.predict(
    endpoint="openai-embeddings-iherb-test",
    inputs={
        "input": "What is the capital of France?",
        "dimensions": 256,
    }
)


## Create a custom pyfunc for external model serving

In [0]:
import mlflow
from openai import OpenAI
import pandas as pd

API_TOKEN = dbutils.secrets.get(scope="scope_name", key="databricks_pat")

class EmbeddingModel(mlflow.pyfunc.PythonModel):
  def __init__(self):
    # Initialize the OpenAI client with the API key from Databricks secrets
    self.client = OpenAI(
        api_key=dbutils.secrets.get(scope = "notebooks", key = "jon_openai_api_key")
    )
  
  def retrieve_embeddings(self, input):
    # Retrieve embeddings for the given input using the OpenAI embeddings API
    embeddings = self.client.embeddings.create(
      model="text-embedding-3-small",
      input=input,
      dimensions=256
    )
  
  def predict(self, context, model_input):
    """
    MLflow calls this method to make predictions. It expects `model_input` to be a pandas DataFrame.
    This method applies the retrieve_embeddings function to each input in the DataFrame
    and returns the embeddings as a new DataFrame.
    """
    # Assuming the input DataFrame has a column named 'input'
    embeddings = model_input['input'].apply(self.retrieve_embeddings)
    # Return as a DataFrame
    return pd.DataFrame({'embedding': embeddings})

In [0]:
import os
from openai import OpenAI

client = OpenAI(
    # This is the default and can be omitted
    api_key=dbutils.secrets.get(scope = "notebooks", key = "jon_openai_api_key")
)

response = client.responses.create(
    model="gpt-4o",
    instructions="You are a coding assistant that talks like a pirate.",
    input="How do I check if a Python object is an instance of a class?",
)

print(response.output_text)